# 03 — Build Feature Table (MLP)
# Giai đoạn 1 — Mục 1.4 — Xây dựng bảng đặc trưng 32 chiều cho MLP

- **Đầu ra**: `outputs/tables/features_mlp.parquet`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from common import io_utils, features_full, config as cfg

In [2]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
manifest_filtered = pd.read_csv(TABLES_DIR / "manifest_filtered.csv")
manifest_filtered.head()

,file_path,load_hp,label,fault_diameter_mils,or_position,source_category,sensor_location,declared_sample_rate_khz,n_samples_DE,n_samples_FE,n_samples_BA,rpm_from_file,read_error,warnings,has_warning
0,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,0,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,122571,122571.0,122571.0,1796.0,NaN,NaN,False
1,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,1,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121410,121410.0,121410.0,1772.0,NaN,NaN,False
2,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,2,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1748.0,NaN,NaN,False
3,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,3,B,7.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121556,121556.0,121556.0,1722.0,NaN,NaN,False
4,..\..\data\raw\12k_Drive_End_Bearing_Fault_Dat...,0,B,14.0,NaN,12k_Drive_End_Bearing_Fault_Data,DE,12.0,121846,121846.0,121846.0,1796.0,NaN,NaN,False


- Đọc cấu hình bandpass từ `bandpass_config.json` (đã chốt ở notebook 02)


In [ ]:
import json
from scipy.signal import resample_poly

# Đọc cấu hình bandpass đã chốt ở notebook 02
with open(TABLES_DIR / "bandpass_config.json") as f:
    bandpass_cfg = json.load(f)
BAND_HZ = tuple(bandpass_cfg["band_hz"])
LP_CUTOFF_HZ = bandpass_cfg["lp_cutoff_hz"]

# --- Xử lý sampling rate không đồng nhất của Normal baseline (GĐ0 mục 0.1.4) ---
# TODO: nên chuyển mapping này vào common/config.py để tránh lặp lại giữa các notebook
TARGET_FS = 12000
NORMAL_FS_OVERRIDE = {
    "97_Normal_0.mat": 24000,
    "98_Normal_1.mat": 48000,
    "99_Normal_2.mat": 48000,
    "100_Normal_3.mat": 48000,
}

def detect_fs(file_path):
    return NORMAL_FS_OVERRIDE.get(Path(file_path).name, TARGET_FS)

def load_de_signal_fixed_fs(file_path, target_fs=TARGET_FS):
    x = io_utils.load_de_signal(Path(file_path))
    fs = detect_fs(file_path)
    if fs != target_fs:
        x = resample_poly(x, target_fs, fs)
    return x


# Xây dựng bảng đặc trưng cho MLP

> ⚠️ **Lưu ý từ GĐ0 (mục 0.4.2):** bể 32 chiều hiện tại quá nhỏ (Feature Selection chọn còn ~30, Δ ~6%).
> Cần mở rộng lên ~50–60 chiều (tăng hài 3→5, thêm skewness/kurtosis/crest factor…) trước khi chạy Feature Selection ở GĐ2.
> Việc mở rộng thực hiện trong `common/features_full.py`.


In [ ]:
# Đồng bộ lp_cutoff với notebook 02 (nếu build_full_feature_table hỗ trợ tham số này)
try:
    feature_df_mlp = features_full.build_full_feature_table(
        manifest_filtered,
        band_hz=BAND_HZ,
        lp_cutoff=LP_CUTOFF_HZ,
        load_de_signal_fn=load_de_signal_fixed_fs,
    )
except TypeError:
    feature_df_mlp = features_full.build_full_feature_table(
        manifest_filtered,
        band_hz=BAND_HZ,
        load_de_signal_fn=load_de_signal_fixed_fs,
    )

# Lưu
feature_df_mlp.to_parquet(TABLES_DIR / "features_mlp.parquet")
print(f"Đã lưu bảng đặc trưng MLP: {len(feature_df_mlp)} dòng")
feature_df_mlp.head()
